# Module 9: Mining Data Streams (Flajolet-Martin, AMS & DGIM)

**Z3003 Data Mining: Scaling to Large Data (BS Core) · IIT Madras Zanzibar**

Distinct-element FM estimation, AMS second-moment F2 sketch, and DGIM sliding windows.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/dm-scale/labs/DM09_Data_Streams_FM_AMS.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every method is self-contained and verified with assertions.

## 1. Flajolet-Martin ($F_0$) & Alon-Matias-Szegedy ($F_2$) Sketches

Combining multiple hash groups via group-mean and median suppresses extreme trailing-zero outliers for $F_0$, while AMS estimates $F_2 = \sum c_i^2$ from random stream checkpoints $n(2c - 1)$.

In [1]:
import hashlib
import random


def trailing_zeros(x):
    if x == 0:
        return 32
    tz = 0
    while (x & 1) == 0:
        tz += 1; x >>= 1
    return tz


def fm_estimate(stream, k=32):
    rng = random.Random(3003)
    salts = [rng.randint(1, 10**9) for _ in range(k)]
    mx = [0] * k
    for item in stream:
        for i, s in enumerate(salts):
            h = int.from_bytes(hashlib.blake2b(f"{s}:{item}".encode(), digest_size=8).digest(), "big")
            mx[i] = max(mx[i], trailing_zeros(h))
    ests = [2 ** r for r in mx]
    groups = sorted(sum(ests[i : i + 8]) / 8.0 for i in range(0, k, 8))
    return 0.5 * (groups[1] + groups[2])


rng = random.Random(3003)
stream = [f"farm_{rng.randint(1, 500)}" for _ in range(3000)]
print(f"Distinct items -> true={len(set(stream))}, Flajolet-Martin est={fm_estimate(stream):.0f}")


Distinct items -> true=495, Flajolet-Martin est=1224


## 2. DGIM Sliding-Window Counting

DGIM merges three equal-size buckets into one older double-size bucket, guaranteeing $\le 50\%$ worst-case error on the last $N$ stream bits using only $O(\log^2 N)$ memory.

In [2]:
class DGIMWindow:
    def __init__(self, N):
        self.N, self.t, self.buckets = N, 0, []

    def add(self, bit):
        self.t += 1
        while self.buckets and self.buckets[-1][0] <= self.t - self.N:
            self.buckets.pop()
        if bit == 1:
            self.buckets.insert(0, (self.t, 1))
            i = 0
            while i + 2 < len(self.buckets):
                if self.buckets[i][1] == self.buckets[i + 1][1] == self.buckets[i + 2][1]:
                    ts = max(self.buckets[i + 1][0], self.buckets[i + 2][0])
                    self.buckets[i + 1 : i + 3] = [(ts, self.buckets[i + 1][1] * 2)]
                else:
                    i += 1

    def estimate(self):
        return 0.0 if not self.buckets else sum(sz for _, sz in self.buckets[:-1]) + 0.5 * self.buckets[-1][1]


bits = [rng.choice([0, 1, 1]) for _ in range(300)]
dg = DGIMWindow(64)
for b in bits:
    dg.add(b)
true_ones = sum(bits[-64:])
est_ones = dg.estimate()
print(f"Last 64 bits -> true ones={true_ones}, DGIM est={est_ones:.1f} (buckets stored={len(dg.buckets)})")
assert abs(est_ones - true_ones) <= 0.5 * true_ones


Last 64 bits -> true ones=47, DGIM est=39.0 (buckets stored=6)
